In [ ]:
# !pip install polars
# !pip install --upgrade pandas-gbq
# !pip install --upgrade pandas google-cloud-bigquery
# !pip install --upgrade fsspec
# !pip install --upgrade scipy
# !pip install --upgrade numpy

In [ ]:
import os
import json
import re
import subprocess
import numpy as np
import pandas as pd
import pandas_gbq
import polars as pl
from google.cloud import bigquery
import pandas_gbq
import warnings
import matplotlib.pyplot as plt
from scipy.stats import norm
import gcsfs

In [ ]:
import matplotlib as mpl
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrices

In [ ]:
#setup of variables
def wb(*args):
    """Run a wb command and return parsed JSON."""
    cmd = ["wb", *args, "--format=json"]
    result = subprocess.check_output(cmd, text=True)
    return json.loads(result)
 
 
# Get workspace info
workspace = wb("workspace", "describe")
GOOGLE_CLOUD_PROJECT = workspace["googleProjectId"]
 
# Get resources
resources = wb("resource", "list")
 
# WORKSPACE_BUCKET
bucket_resources = [
    r for r in resources
    if r.get("resourceType") == "GCS_BUCKET"
    and "practical_considerations_bucket" in r.get("id", "")
    and "temporary" not in r.get("id", "")
]
 
if not bucket_resources:
    raise ValueError("No matching bucket found")
 
WORKSPACE_BUCKET = f"gs://{bucket_resources[0]['bucketName']}"
 
# WORKSPACE_CDR
bq_resources = [
    r for r in resources
    if r.get("resourceType") in {"BQ_DATASET", "BIGQUERY_DATASET"}
]
 
cdr_resources = [
    r for r in bq_resources
    if re.match(r"^C\d{4}Q\d+R\d+$", r.get("datasetId", ""))
]
 
if not cdr_resources:
    raise ValueError("No matching CDR dataset found")
 
WORKSPACE_CDR = (
    f"{cdr_resources[0]['projectId']}."
    f"{cdr_resources[0]['datasetId']}"
)


In [ ]:
#get variables
version = WORKSPACE_CDR
bucket = WORKSPACE_BUCKET
cohort = "allofus"

In [ ]:
os.environ["WORKSPACE_CDR"] = version

In [ ]:
ancestry_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv"

In [ ]:
google_project_id = %env GOOGLE_CLOUD_PROJECT

In [ ]:
client = bigquery.Client(project=google_project_id)

def download_data(query):
    return client.query(query).to_dataframe()

In [ ]:
query = f"""
SELECT  person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name AS gender,
        person.birth_datetime AS date_of_birth,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name AS sex_at_birth,
        meas.measurement_source_concept_id,
        meas.value_as_number,
        meas.measurement_datetime,
        -- Age (years) at the time of measurement
        DATETIME_DIFF(
            DATETIME(meas.measurement_datetime),
            DATETIME(person.birth_datetime),
            DAY
        ) / 365.2425 AS natural_age
FROM `{WORKSPACE_CDR}.person` person
INNER JOIN `{WORKSPACE_CDR}.measurement` meas
        ON person.person_id = meas.person_id
LEFT JOIN `{WORKSPACE_CDR}.concept` p_gender_concept
        ON person.gender_concept_id = p_gender_concept.concept_id
LEFT JOIN `{WORKSPACE_CDR}.concept` p_sex_at_birth_concept
        ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id
LEFT JOIN `{WORKSPACE_CDR}.death` d
        ON person.person_id = d.person_id
WHERE meas.measurement_source_concept_id IN (903133)
  -- keep only measurements taken while the person was alive
  AND (d.death_date IS NULL
       OR DATE(meas.measurement_datetime) <= d.death_date)
"""

person_df = download_data(query)

In [ ]:
print(f"N participants: {len(person_df)}")

In [ ]:
# Rename value_as_number to height_cm
person_df = person_df.rename(columns={"value_as_number": "height_cm"})

# Compute median height per person, keeping sex_at_birth
height_df = (
    person_df.groupby(["person_id", "sex_at_birth"])
    .agg(
        median_height=("height_cm", "median"),
        natural_age=("natural_age", "median"),
    )
    .reset_index()
)

# Confirm no duplicate person_id
assert height_df["person_id"].is_unique, "Duplicate person_id found!"

In [ ]:
prs_df = pl.read_csv(f'{bucket}/mhdata/Height_output/score/Height_final_scores.csv')

In [ ]:
prs_df = prs_df.rename({"sample_id": "person_id"})

In [ ]:
fs = gcsfs.GCSFileSystem(project=google_project_id, requester_pays=True)
with fs.open(ancestry_path, "rb") as f:
    ancestry_df = pl.read_csv(
        f,
        separator="\t")
#split PCs into columns
n_pcs = 16
pc_names = [f"PC{i+1}" for i in range(n_pcs)]

ancestry_df = (
    ancestry_df
    .with_columns(
        pl.col("pca_features")
          .str.strip_chars("[]")
          .str.split(",")
          .list.eval(pl.element().str.strip_chars().cast(pl.Float64))
          .list.to_struct(fields=pc_names)
          .alias("pcs")
    )
    .unnest("pcs")
    .drop("pca_features")
)

In [ ]:
# Cast research_id to integer to match person_id
prs_anc_df = prs_df.join(
    ancestry_df.with_columns(pl.col('research_id').cast(pl.Int64)),
    left_on="person_id",
    right_on="research_id",
    how="left"
)

In [ ]:
prs_anc_df = prs_anc_df.with_columns(
    ((pl.col("sum_weights") - pl.col("sum_weights").mean()) / pl.col("sum_weights").std()).alias("sum_weights_scaled")
)

In [ ]:
prs_anc_df = prs_anc_df.to_pandas()

In [ ]:
prs_anc_df["person_id"] = prs_anc_df["person_id"].astype("int64")

In [ ]:
df_merged = height_df.merge(prs_anc_df, on="person_id", how="inner")  
df_merged = df_merged.dropna(subset=["median_height", "sum_weights", "ancestry_pred", "sex_at_birth"])
df_merged = df_merged[df_merged["sex_at_birth"].isin(["Male", "Female"])]
# Check duplicates
print("duplicate person_ids:", df_merged["person_id"].duplicated().sum())
df_merged = df_merged.drop_duplicates("person_id")
# Deciles within ancestry
df_merged["sum_weights_decile"] = (
    df_merged.groupby(["ancestry_pred", "sex_at_birth"])["sum_weights"]
    .transform(lambda x: pd.qcut(x, 10, labels=False, duplicates="drop") + 1)
)
# Verify: rows should be nearly identical across columns
print(df_merged.groupby(["ancestry_pred","sex_at_birth", "sum_weights_decile"]).size().unstack())

In [ ]:
correlation_SR = df_merged[["median_height", "sum_weights"]].dropna().corr(method="pearson").iloc[0, 1]

In [ ]:
df_merged["ancestry_pred"].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Scatterplot
ax.scatter(df_merged["sum_weights"], df_merged["median_height"], alpha=0.6)

# Regression line
mask = df_merged[["sum_weights", "median_height"]].dropna()
slope, intercept = np.polyfit(mask["sum_weights"], mask["median_height"], 1)
x_vals = np.array([mask["sum_weights"].min(), mask["sum_weights"].max()])
y_vals = slope * x_vals + intercept
ax.plot(x_vals, y_vals, color="blue")

# Correlation coefficient annotation
ax.text(20, 70, f"pearson r2 = {round(correlation_SR, 2)}",
        color="black", fontsize=12, ha="left")

# Labels and title
ax.set_xlabel("PRS")
ax.set_ylabel("Self Report + Exam")
ax.set_title("Height vs Height Risk Scores")

plt.tight_layout()
plt.show()

In [ ]:
mpl.rcParams["pdf.fonttype"] = 42 

# Keep only Male / Female
df_sex = df_merged[df_merged["sex_at_birth"].isin(["Female", "Male"])]

df_summary_split = (
    df_sex.groupby(["sex_at_birth", "sum_weights_decile", "ancestry_pred"])["median_height"]
    .agg(
        mean_height="mean",
        sem=lambda x: x.std() / np.sqrt(x.count()),
        n="count",
    )
    .reset_index()
)

fill_map = {"afr": "#ff8a42", "eur": "#85a7ff", "amr": "#92deb4", "mid": "#fc72b6", "eas": "#bcadff", "sas": "#ffbd2b"}
color_map = {"afr": "#FE6100", "eur": "#648FFF", "amr": "#77C89B", "mid": "#DC267F", "eas": "#785ef0", "sas": "#FFB000"}

fig, ax = plt.subplots(figsize=(9, 7))

for (sex, ancestry), group in df_summary_split.groupby(["sex_at_birth", "ancestry_pred"]):
    group = group.sort_values("sum_weights_decile")

    # Line (label only once per ancestry so the legend isn't duplicated)
    ax.plot(group["sum_weights_decile"], group["mean_height"],
            color=color_map[ancestry], linewidth=2.5,
            label=ancestry if sex == "Male" else None, zorder=2)

    # Points (white edge, colored fill)
    ax.scatter(group["sum_weights_decile"], group["mean_height"],
               facecolor=fill_map[ancestry], edgecolor="white", linewidth=1.5,
               s=80, zorder=3)

    # Error bars
    ax.errorbar(group["sum_weights_decile"], group["mean_height"],
                yerr=group["sem"], fmt="none", ecolor="black",
                capsize=4, elinewidth=1.2, zorder=5)

ax.set_xticks(range(1, 11))

ax.set_xlabel("Height Polygenic Risk Score Decile", fontsize=16, fontweight="bold", labelpad=20)
ax.set_ylabel("Height (cm), mean (SE)", fontsize=16, fontweight="bold", labelpad=20)

ax.tick_params(axis="both", labelsize=14, color="black", width=1.2, length=0.25 * 72)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight("bold")
    label.set_color("black")

# Classic theme: only bottom/left spines visible
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_linewidth(1.0)
    ax.spines[s].set_color("black")

# Smaller legend
legend = ax.legend(
    title="Genetic Ancestry", loc="upper left", bbox_to_anchor=(0.03, 0.99),
    frameon=True, fontsize=9, title_fontsize=10,
    handlelength=1.2, handletextpad=0.5, labelspacing=0.3,
    borderpad=0.4, borderaxespad=0.3,
)
legend.get_frame().set_facecolor("white")
legend.get_frame().set_edgecolor("black")
for text in legend.get_texts():
    text.set_fontweight("bold")
legend.get_title().set_fontweight("bold")

plt.tight_layout()
plt.savefig("height_by_prs_decile_anc_by_sex.pdf", format="pdf", bbox_inches="tight")
plt.savefig("height_by_prs_decile_anc_by_sex.png", format="png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Decile split on height
d1_cutoff = df_merged["median_height"].quantile(0.10)
d10_cutoff = df_merged["median_height"].quantile(0.90)

df_extremes = df_merged[
    (df_merged["median_height"] <= d1_cutoff) | (df_merged["median_height"] >= d10_cutoff)
].copy()

df_extremes["height_decile"] = np.where(
    df_extremes["median_height"] <= d1_cutoff, "Bottom Decile", "Top Decile"
)

color_map = {"Bottom Decile": "#E9D0E7", "Top Decile": "#7F3B7A"}

fig, ax = plt.subplots(figsize=(3.2, 2.6))

for decile, group in df_extremes.groupby("height_decile"):
    x = group["sum_weights"].dropna()  # PRS distribution, split by height decile
    mu, sigma = x.mean(), x.std()
    xs = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
    ys = norm.pdf(xs, mu, sigma)
    ax.plot(xs, ys, color=color_map[decile], linewidth=3)
    ax.axvline(mu, color=color_map[decile], linewidth=3, linestyle=(0, (4, 3)))

ax.set_xlabel("Genetic Risk", fontsize=24, color="black")
ax.set_ylabel("Density", fontsize=24, color="black")
ax.set_xticks([])
ax.set_yticks([])

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(2)
ax.spines["bottom"].set_linewidth(2)
ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")

plt.tight_layout()
plt.savefig("prs_by_height_top_bottom_deciles.pdf", format="pdf", dpi=300, bbox_inches="tight")
plt.show()

# Regression calculation

In [ ]:
r_df = height_df.merge(prs_anc_df, on="person_id", how="left")

In [ ]:
r_df.head()

In [ ]:
keep = ["person_id", "sex_at_birth", "natural_age", "median_height", "sum_weights", "ancestry_pred", "PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7", "PC8", "PC9", "PC10"]
r_df = r_df[keep].copy()

In [ ]:
#create subset to male and female r^2
fr_df = r_df[r_df["sex_at_birth"] == "Female"]
mr_df = r_df [r_df["sex_at_birth"] == "Male"]

## No PRS in formula

### Subset to male 

In [ ]:
model = smf.ols("median_height ~ natural_age + sex_at_birth + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10", data=mr_df).fit()
print(model.rsquared)

In [ ]:
formula = ("median_height ~ natural_age + sex_at_birth + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10")

rows = []
for anc, sub in mr_df.groupby("ancestry_pred"):
    model = smf.ols(formula, data=sub).fit()
    rows.append({
        "ancestry_pred": anc,
        "n": int(model.nobs),
        "r2": model.rsquared
    })

r2_df = pd.DataFrame(rows)
print(r2_df)

### Subset to female 

In [ ]:
model = smf.ols("median_height ~ natural_age + sex_at_birth + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10", data=fr_df).fit()
print(model.rsquared)

In [ ]:
formula = ("median_height ~ natural_age + sex_at_birth + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10")

rows = []
for anc, sub in fr_df.groupby("ancestry_pred"):
    model = smf.ols(formula, data=sub).fit()
    rows.append({
        "ancestry_pred": anc,
        "n": int(model.nobs),
        "r2": model.rsquared
    })

r2_df = pd.DataFrame(rows)
print(r2_df)

## PRS in formula

### Subset to male

In [ ]:
#every male
model_prs = smf.ols("median_height ~ natural_age + sex_at_birth + sum_weights + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10", data=mr_df).fit()
print(model.rsquared)

In [ ]:
formula2 = ("median_height ~ natural_age + sex_at_birth + sum_weights + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10")

rows = []
for anc, sub in mr_df.groupby("ancestry_pred"):
    model = smf.ols(formula2, data=sub).fit()
    rows.append({
        "ancestry_pred": anc,
        "n": int(model.nobs),
        "r2": model.rsquared
    })

r2_df = pd.DataFrame(rows)
print(r2_df)

### Subset to female

In [ ]:
#every female
model_prs = smf.ols("median_height ~ natural_age + sex_at_birth + sum_weights + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10", data=fr_df).fit()
print(model.rsquared)

In [ ]:
formula2 = ("median_height ~ natural_age + sex_at_birth + sum_weights + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10")

rows = []
for anc, sub in fr_df.groupby("ancestry_pred"):
    model = smf.ols(formula2, data=sub).fit()
    rows.append({
        "ancestry_pred": anc,
        "n": int(model.nobs),
        "r2": model.rsquared
    })

r2_df = pd.DataFrame(rows)
print(r2_df)